In [1]:
import preprocess as pp 
import calibration as cb 
import shortestpath as sp 
import pandas as pd
import geopandas as gpd
from shapely.ops import unary_union
from shapely.geometry import Polygon, MultiPolygon
import os

In [2]:
#Select Area
a = 0
#Foldername
case = f'kochi{a}'
#Load data
kochi_evacbldg = gpd.read_file('./kochi_data/kochi_tsunami_evacbldg_crs4326.geojson')
kochi_shelters = gpd.read_file('./kochi_data/kochi_tsunami_shelters_crs4326.geojson')
kochi_inund = gpd.read_file('./kochi_data/kochi-shi_tsunami_inundation_crs4326.geojson')
kochi_census = gpd.read_file('./kochi_data/kochi-shi_census_crs4326.geojson')
kochi_areas = gpd.read_file('./kochi_data/kochi-shi_tsunamievac_areas_crs4326.geojson')

In [3]:
no_of_areas = kochi_areas.geometry.size
print("Number of Evacuation Areas: ", no_of_areas)
#Create a dictionary of areas
areas = {}
for i, area in enumerate(kochi_areas.geometry):
    areas[f'kochi{i}'] = area

Number of Evacuation Areas:  6


In [4]:
def convert_multipolygon_to_polygon(multipolygon = areas['kochi0']):
    # Merge polygons into a single geometry
    merged = unary_union(multipolygon)

    # Check result type and convert to Polygon if possible
    if merged.geom_type == 'Polygon':
        polygon = merged
    elif merged.geom_type == 'MultiPolygon':
        # Choose the largest polygon by area
        polygon = max(merged.geoms, key=lambda p: p.area)

    # Verify the result
    print(type(polygon))
    return polygon

# iterate over areas to convert their geometries to polygons
for i, area in enumerate(kochi_areas.geometry):
    if area.geom_type == 'MultiPolygon':
        areas[f'kochi{i}'] = convert_multipolygon_to_polygon(area)
    else:
        areas[f'kochi{i}'] = area

<class 'shapely.geometry.polygon.Polygon'>
<class 'shapely.geometry.polygon.Polygon'>
<class 'shapely.geometry.polygon.Polygon'>
<class 'shapely.geometry.polygon.Polygon'>
<class 'shapely.geometry.polygon.Polygon'>
<class 'shapely.geometry.polygon.Polygon'>


In [5]:
#Merge all building data
merged_gdf = gpd.GeoDataFrame(pd.concat([kochi_evacbldg, kochi_shelters], ignore_index=True))

In [6]:
import osmnx as ox

def get_graph(polygon):
    G = ox.graph_from_polygon(polygon, network_type='all', simplify=True)
    return G

def get_osmids(G, gdf):
    # Step 1: Get features from "merged_gdf" contained within "G" bounding box
    # Convert the graph to GeoDataFrames
    nodes, edges = ox.graph_to_gdfs(G)

    # Get the total bounds of the edges GeoDataFrame
    G_bbox = edges.total_bounds
    #Find shelters in G area
    shelters_in_G = gdf.cx[G_bbox[0]:G_bbox[2], G_bbox[1]:G_bbox[3]]

    # Step 2: Identify the closest node in G for each feature in 'shelters_in_G'
    # shelters_in_G['closest_node'] 
    osmid_list = shelters_in_G.geometry.apply(lambda point: ox.distance.nearest_nodes(G, point.x, point.y))
    return osmid_list

In [7]:
#Create shleters by area
shelters = {}
for i, area in enumerate(areas):
    G = get_graph(areas[f'kochi{i}'])
    shelters[f'kochi{i}'] = get_osmids(G, merged_gdf).to_list()

In [8]:
def calculate_population(area, gdf):
    # Filter the census data to only include rows within the given area
    within_area = gdf[gdf.geometry.within(area)]
    # Sum the population column for the filtered rows
    return within_area['M_TOTPOP_H'].sum()

def calculate_population_for_areas(areas, gdf):
    populations = {}
    for area_name, area_geom in areas.items():
        populations[area_name] = calculate_population(area_geom, gdf)
    return populations

# Calculate total population for each area
pop = calculate_population_for_areas(areas, kochi_census)
pop

{'kochi0': np.int64(2303),
 'kochi1': np.int64(1078),
 'kochi2': np.int64(622),
 'kochi3': np.int64(240),
 'kochi4': np.int64(13244),
 'kochi5': np.int64(0)}

In [9]:
times_population = [1]
pp.main(areas=areas, 
        foldername=case, 
        evacnodes=shelters[case], 
        times=times_population, 
        pop=pop[case], 
        multi=True, #looks like always need to be true
        use_seed=False, 
        sp=True
        ) 

rm: ./kochi0_CAREFUL_PREVIOUS_INPUT: No such file or directory


Folder kochi0 created
Graph downloaded
Node 6290324637 is an evacuation node
Node 9935063198 is an evacuation node
Node 4942039738 is an evacuation node
Node 5447091880 is an evacuation node
Node 9935063192 is an evacuation node
Node 4942039753 is an evacuation node
Short links removed, nodes merged, counters updated, and link references adjusted successfully.
Population in file: 2303


In [ ]:
# Run Shortest Path
simulTime = 120*60
meanrayleigh = 5
numExperiments = 100
results_df = []
results_time = []

for i in range(numExperiments):
    survivedAgents_df, evactime = sp.run(foldername=case, timeSimulation=simulTime, popfile=1, meanrayleigh=meanrayleigh, video=False)
    results_df.append(survivedAgents_df)
    results_time.append(evactime)
    print(f"Experiment {i+1}/{numExperiments} completed.")

In [ ]:
cv = pd.DataFrame({'time':results_time}).expanding(2).std() / pd.DataFrame({'time':results_time}).expanding(2).mean()
cv.plot()

---

In [ ]:
import matplotlib.pyplot as plt

# Add a new column 'safe_rate'
survivedAgents_df['safe_rate'] = survivedAgents_df['safe'] / pop[case]

# Plot 'safe_rate' against 'time'
plt.figure(figsize=(10, 6))
plt.plot(survivedAgents_df['time'], survivedAgents_df['safe_rate'], label='Safe Rate')
#make a verticl line at evactime
plt.axvline(x=evactime, color='r', linestyle='--', label='Evacuation Time')
plt.xlabel('Time')
plt.ylabel('Safe Rate')
plt.title('Safe Rate Over Time')
plt.legend()
plt.grid()
plt.show()

In [ ]:
#create layers for mapping
point_layers = []
evacbldg_layer = {
    'gdf': kochi_evacbldg,
    'name': 'Evacuation Buildings',
    'color': 'blue',
    'icon' : 'building'
}
point_layers.append(evacbldg_layer)
shelter_layer = {
    'gdf': kochi_shelters,
    'name': 'Shelters',
    'color': 'green',
    'icon' : 'home'
}
point_layers.append(shelter_layer)

polygon_layers = []
inund_layer = {
    'gdf': kochi_inund,
    'name': 'Inundation Area',
    'color': 'blue',
}
polygon_layers.append(inund_layer)
census_layer = {
    'gdf': kochi_census,
    'name': 'Population',
    'color': 'red',
}
polygon_layers.append(census_layer)
areas_layer = {
    'gdf': kochi_areas,
    'name': 'Evacuation Areas',
    'color': 'yellow',
}
polygon_layers.append(areas_layer)

In [ ]:
pp.create_html_map(point_layers, polygon_layers)

In [ ]:
ox.plot_graph(G6, node_size=10, node_color='black', edge_color='black', edge_linewidth=0.5, bgcolor='white', show=False, close=False)


In [ ]:
kochi_areas